In [1]:
import numpy as np
import matplotlib.pyplot as plt
from time import sleep
from pydaqmx_helper.dac import DAC

In [2]:
# Defining path to folder raw data is saved in
path = "C:/Users/Student/Desktop/Robert-Kelly-Nuvistor-Triode/"

In [3]:
myDAC = DAC(0)
myDAC.writeVoltage(2.5)

#### Setting up the picoammeter

In [4]:
# Import pyserial and set up a connection to the picoammeter.
# # [From physicslabs.ucd.ie RBD 910 Picoammeter documentation.]
import serial

ser = serial.Serial(
    "com5",
    baudrate = 57600,                    # Sets the bits per second to 57600
    bytesize = serial.EIGHTBITS,         # Sets the data bits to 8
    parity = serial.PARITY_NONE,         # Sets the parity to None
    stopbits = serial.STOPBITS_ONE,      # Sets the stop bits to 1
    xonxoff = False,                     # Sets the flow control to None
    timeout = 1,                         # Sets the timeout to 1 second
)

In [5]:
# Must talk to the picoammeter in arrays of bytes.
# data() turns a string "command" to be sent to the picoammeter into bytes 
# and also ends the string with the newline character "\n" so it does not have to be done manually
# # [From physicslabs.ucd.ie RBD 910 Picoammeter documentation.]

def data(command):
    return bytes(command+"\n", "utf-8")

def communicate(command): # This communicates the command to the picoammeter and returns its answer.
    ser.write(data(command))
    output = ser.readlines()
    print(output)

In [6]:
# Explicitly set filter.
ser.write(data("&F064"))
ser.readlines()

[b'&F, Filter=064\r\n']

In [7]:
ser.write(data("&S"))
ser.readlines()

[b'&S=,Range=002nA,-0.0027,nA\r\n']

#### Define range of voltages to be sent to the circuit

The USB-6008 ranges from 0-5V. We want it to range from negative voltages to positive voltages so the NIDAQ analogue output adapter changes an input of 0V to -5V, 2.5V to 0V, and 5V to 5V.

In [8]:
# Write function to convert the actual voltage we want in the circuit to the value we tell the USB-6008 to transmit.
def convert_voltage(V):
    return V/2 + 2.5

#### Create function to wrangle current measurement into a list of strings


In [13]:
def communicate(command):       # Only valid for commands which return a single line from the picoammeter
    ser.write(data(command))    # When the command is "&S", the picoammeter gives back a list with a single byte inside.
    output = ser.readline()                  # Let output be that list with the single byte.    

    output = output.decode("utf-8").rstrip() # decode the byte into a string and strip the newline character
    output = output.split(",")               # split the string into a list of 4 strings
    
    return output

#### Create function that takes data and saves it for me.

In [31]:
def collect_data(V_A_circuit, delay, I_F):
    # V_A_circuit = The array of applied anode voltages actually experienced by the circuit.
    # Delay =       The time in seconds that the code will be delayed before moving to the next voltage in the range.
    # I_F =         The filament current.

    # V_A is the array of applied voltages given to the USB-6008 so that V_A_circuit is actually produced in the circuit.
    V_A = convert_voltage(V_A_circuit)

    # Specify to send voltages using channel 0 on the USB-6008
    myDAC = DAC(0)

    # empty array for recording the anode currents I_A, and their errors.
    I_A = np.array([])
    error_I_A = np.array([])

    # Save all outputs for analysis
    alloutputs = []
    
    # Loop over all voltages in the specified range.
    for V in V_A:

        # Send each voltage in the range to the circuit via the USB-6008
        myDAC.writeVoltage(V)

        if V == V_A[0]: # Sleep for longer before the first measurement
            sleep(10)


        #After each voltage in, wait before moving onto the next voltage in the range.
        sleep(delay)
        
        # Measure current from the picoammeter.
        output = communicate("&S")
        
        #Count number of repetitions needed to get stable measurement. 
        repetitions = 0
        
        print("index = %.f" % np.where(V_A == V) , output[0])
        
         # If picoammeter gives out a bad measurement, print what index of V_A it is at, and 
         # if it over-ranges, under-ranges, or is unstable
        while output[0] != '&S=' and repetitions < 60: # Repeat measurements until the picoammeter gives a stable measurement
                                                        # that is not over-range or under-range.
                                                      # Break loop if repetitions gets very high.                        
            sleep(delay * 2.5)

            output = communicate("&S")

            repetitions += 1

            print(repetitions) 
        
        
        alloutputs = np.append(alloutputs, output)

        # Extracting the current from the output and accounting for the scale.
        if output[3] == "nA":
            # Then the third element of output is the current in nano-amps
            current = float(output[2]) * 10**-9
            error_current = 0.00005 * 10**-9

        if output[3] == "uA":
            # Then the third element of output is the current in micro-amps
            current = float(output[2]) *10**-6
            error_current = 0.00005 * 10**-6

        if output[3] == "mA":
            # Then the third element of output is the current in milli-amps
            current = float(output[2]) *10**-3
            error_current = 0.00005 * 10**-3

        I_A = np.append(I_A, current)
        error_I_A = np.append(error_I_A, error_current)
        print(output)
    myDAC.writeVoltage(convert_voltage(0))

    # Save raw data to a file so it can be accessed later.
    np.save(path+f"anode currents for I_F = {I_F} mA", I_A)
    np.save(path+f"anode voltages for I_F = {I_F} mA", V_A_circuit)
    np.save(path+f"alloutputs for I_F = {I_F} mA", alloutputs)
    np.save(path+f"anode currents errors for I_F = {I_F} mA", error_I_A)

In [30]:
collect_data(np.arange(-0.5, 1.5, .005), 2, 90)

index = 0 &S=
['&S=', 'Range=002nA', '-0.0010', 'nA']
index = 1 &S=
['&S=', 'Range=002nA', '-0.0010', 'nA']
index = 2 &S=
['&S=', 'Range=002nA', '-0.0010', 'nA']
index = 3 &S=
['&S=', 'Range=002nA', '-0.0011', 'nA']
index = 4 &S=
['&S=', 'Range=002nA', '-0.0011', 'nA']
index = 5 &S=
['&S=', 'Range=002nA', '-0.0009', 'nA']
index = 6 &S=
['&S=', 'Range=002nA', '-0.0009', 'nA']
index = 7 &S=
['&S=', 'Range=002nA', '-0.0010', 'nA']
index = 8 &S=
['&S=', 'Range=002nA', '-0.0011', 'nA']
index = 9 &S=
['&S=', 'Range=002nA', '-0.0010', 'nA']
index = 10 &S=
['&S=', 'Range=002nA', '-0.0010', 'nA']
index = 11 &S=
['&S=', 'Range=002nA', '-0.0011', 'nA']
index = 12 &S=
['&S=', 'Range=002nA', '-0.0010', 'nA']
index = 13 &S=
['&S=', 'Range=002nA', '-0.0010', 'nA']
index = 14 &S=
['&S=', 'Range=002nA', '-0.0012', 'nA']
index = 15 &S=
['&S=', 'Range=002nA', '-0.0010', 'nA']
index = 16 &S=
['&S=', 'Range=002nA', '-0.0011', 'nA']
index = 17 &S=
['&S=', 'Range=002nA', '-0.0012', 'nA']
index = 18 &S=
['&S=

index = 149 &S=
['&S=', 'Range=020nA', '-03.243', 'nA']
index = 150 &S=
['&S=', 'Range=020nA', '-03.506', 'nA']
index = 151 &S=
['&S=', 'Range=020nA', '-03.788', 'nA']
index = 152 &S=
['&S=', 'Range=020nA', '-04.094', 'nA']
index = 153 &S=
['&S=', 'Range=020nA', '-04.416', 'nA']
index = 154 &S=
['&S=', 'Range=020nA', '-04.782', 'nA']
index = 155 &S=
['&S=', 'Range=020nA', '-05.188', 'nA']
index = 156 &S=
['&S=', 'Range=020nA', '-05.622', 'nA']
index = 157 &S=
['&S=', 'Range=020nA', '-06.091', 'nA']
index = 158 &S=
['&S=', 'Range=020nA', '-06.587', 'nA']
index = 159 &S=
['&S=', 'Range=020nA', '-07.120', 'nA']
index = 160 &S=
['&S=', 'Range=020nA', '-07.700', 'nA']
index = 161 &S=
['&S=', 'Range=020nA', '-08.311', 'nA']
index = 162 &S=
['&S=', 'Range=020nA', '-09.013', 'nA']
index = 163 &S=
['&S=', 'Range=020nA', '-09.797', 'nA']
index = 164 &S=
['&S=', 'Range=020nA', '-10.615', 'nA']
index = 165 &S=
['&S=', 'Range=020nA', '-11.513', 'nA']
index = 166 &S=
['&S=', 'Range=020nA', '-12.470'

index = 296 &S=
['&S=', 'Range=200uA', '-145.88', 'uA']
index = 297 &S=
['&S=', 'Range=200uA', '-148.66', 'uA']
index = 298 &S=
['&S=', 'Range=200uA', '-151.05', 'uA']
index = 299 &S=
['&S=', 'Range=200uA', '-152.95', 'uA']
index = 300 &S=
['&S=', 'Range=200uA', '-154.48', 'uA']
index = 301 &S=
['&S=', 'Range=200uA', '-155.69', 'uA']
index = 302 &S=
['&S=', 'Range=200uA', '-156.66', 'uA']
index = 303 &S=
['&S=', 'Range=200uA', '-157.43', 'uA']
index = 304 &S=
['&S=', 'Range=200uA', '-158.05', 'uA']
index = 305 &S=
['&S=', 'Range=200uA', '-158.59', 'uA']
index = 306 &S=
['&S=', 'Range=200uA', '-159.04', 'uA']
index = 307 &S=
['&S=', 'Range=200uA', '-159.41', 'uA']
index = 308 &S=
['&S=', 'Range=200uA', '-159.72', 'uA']
index = 309 &S=
['&S=', 'Range=200uA', '-159.98', 'uA']
index = 310 &S=
['&S=', 'Range=200uA', '-160.20', 'uA']
index = 311 &S=
['&S=', 'Range=200uA', '-160.39', 'uA']
index = 312 &S=
['&S=', 'Range=200uA', '-160.55', 'uA']
index = 313 &S=
['&S=', 'Range=200uA', '-160.70'

In [19]:
print(I_A)

NameError: name 'I_A' is not defined

In [ ]:
print(alloutputs)

In [ ]:
def unconvert(V_A):
    return 2*V_A - 5

In [ ]:
plt.plot(V_A_circuit, -I_A, "o")
plt.xlabel("Applied anode voltage (V)", fontsize = 12)
plt.ylabel("Anode current ( A)", fontsize = 12)
plt.title("Anode current versus applied anode voltage. ", fontsize = 12);
#plt.xlim(1,1.5);
#plt.ylim(1*10**-5,1.6*10**-5);

In [ ]:
I_AS = 1.5*10**-4
plt.plot(V_A_circuit, np.log10(-I_A/I_AS), "o")
plt.xlabel("Applied anode voltage (V)", fontsize = 12)
plt.ylabel("Anode current", fontsize = 12)
plt.title("I-V curve of anode current versus applied anode voltage in the nuvistor triode", fontsize = 12);

Original working data collection cell.

In [ ]:
# Independent variable
I_F = 80 #mA

# V_A_circuit is the array of applied anode voltages actually experienced by the circuit.
V_A_circuit = np.arange(-0.5, 1.5, .005)

# V_A is the array of applied voltages given to the USB-6008 so that V_A_circuit is actually produced in the circuit.
V_A = convert_voltage(V_A_circuit)

# Specify to send voltages using channel 0 on the USB-6008
myDAC = DAC(0)

# The time in seconds that the code will be delayed before moving to the next voltage in the range.
delay = 1

# empty array for recording the anode currents I_A, and their errors.
I_A = np.array([])
error_I_A = np.array([])

# Save all outputs for analysis
alloutputs = []

# Loop over all voltages in the specified range.
for V in V_A:
    
    # Send each voltage in the range to the circuit via the USB-6008
    myDAC.writeVoltage(V)
    
    if V == V_A[0]: # Sleep for longer before the first measurement
        sleep(10)
        
    
    #After each voltage in, wait before moving onto the next voltage in the range.
    sleep(delay)
    
    # Measure current from the picoammeter
    ser.write(data("&S"))    # When the command is "&S", the picoammeter gives back a list with a single byte inside.
    output = ser.readline()                  # Let output be that list with the single byte.    
    
    alloutputs = np.append(alloutputs, output)
    
    output = output.decode("utf-8").rstrip() # decode the byte into a string and strip the newline character
    output = output.split(",")               # split the string into a list of 4 strings
    
    #Count number of repetitions needed to get stable measurement. 
    repetitions = 0
    
    if repetitions < 60:     # break loop if repetitions gets very high
        
        while output[0] !== '&S': # Repeat measurements until the picoammeter gives a stable measurement
                                  # that is not over-range or under-range

            sleep(delay)

            ser.write(data("&S"))   
            output = ser.readline()                 

            alloutputs = np.append(alloutputs, output)

            output = output.decode("utf-8").rstrip()
            output = output.split(",")

            repetitions += 1

            print(number_unstable_points) 
    
    # Extracting the current from the output and accounting for the scale.
    if output[3] == "nA":
        # Then the third element of output is the current in nano-amps
        current = float(output[2]) * 10**-9
        error_current = 0.00005 * 10**-9
        
    if output[3] == "uA":
        # Then the third element of output is the current in micro-amps
        current = float(output[2]) *10**-6
        error_current = 0.00005 * 10**-6

    if output[3] == "mA":
        # Then the third element of output is the current in milli-amps
        current = float(output[2]) *10**-3
        error_current = 0.00005 * 10**-3
    
    I_A = np.append(I_A, current)
    error_I_A = np.append(error_I_A, error_current)
myDAC.writeVoltage(convert_voltage(0))

# Save raw data to a file so it can be accessed later.
np.save(path+"anode currents for I_F = %f mA" %I_F, I_A)
np.save(path+"anode voltages for I_F = %f" %I_F, V_A_circuit)
np.save(path+"alloutputs for I_F = %f" %I_F, alloutputs)
np.save(path+"anode currents errors for I_F = %f" %I_F, error_I_A)